In [1]:
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")

import numpy as np
import pandas as pd
import h5py

import pycbc.conversions, pycbc.distributions, pycbc.waveform, pycbc.filter, pycbc.types, pycbc.psd, pycbc.fft

from tqdm import tqdm
import datetime
import multiprocessing
import uuid
from argparse import ArgumentParser
import logging

class GenWaveform(object):
    '''Waveform Generator
    '''
    def __init__(self, buffer_length, sample_rate, f_lower):
        self.f_lower = f_lower
        self.delta_f = 1.0 / buffer_length
        tlen = int(buffer_length * sample_rate) # buffer length x sample_rate
        self.flen = tlen // 2 + 1

        #psd is hard coded to O3 psd
        psd = pycbc.psd.read.from_txt('/work/yifanwang/ecc/templatebank/o3psd.txt', 
            self.flen, self.delta_f, self.f_lower, is_asd_file = False)
        
        self.kmin = int(f_lower * buffer_length)
        self.w = ((1.0 / psd[self.kmin:-1]) ** 0.5).astype(np.float32)
        
        qtilde = pycbc.types.zeros(tlen, np.complex64) # correlation in Fourier domain
        q = pycbc.types.zeros(tlen, np.complex64) # correlation
        self.qtilde_view = qtilde[self.kmin:self.flen - 1]
        self.ifft = pycbc.fft.IFFT(qtilde, q)
        
        # the maximum is around 0
        self.md = q._data[-100:]
        self.md2 = q._data[0:100] 

    def generate(self, **kwds):
        '''Return normalized hp
        '''
        if kwds['approximant'] in pycbc.waveform.fd_approximants():  
            hp, _ = pycbc.waveform.get_fd_waveform(delta_f = self.delta_f, **kwds)
        else:
            dt = 1.0 / self.sample_rate
            hp = pycbc.waveform.get_waveform_filter(
                        pycbc.types.zeros(self.flen, dtype=np.complex64),
                        delta_f=self.delta_f,
                        delta_t=dt,
                        f_lower=self.f_lower,
                        **kwds)
        
        hp.resize(self.flen)
        hp = hp.astype(np.complex64)
        
        hp[self.kmin:-1] *= self.w
        s = pycbc.filter.sigmasq(hp, low_frequency_cutoff=self.f_lower)
        hp /= s**0.5 
        
        hp.params = kwds
        hp.s = s

        return hp

    def match(self, hp, hc):
        hp.view = hp[self.kmin:-1]
        hc.view = hc[self.kmin:-1]
        pycbc.filter.correlate(hp.view, hc.view, self.qtilde_view)
        self.ifft.execute()
        m = max(abs(self.md).max(), abs(self.md2).max())
        return m * 4.0 * self.delta_f

    def overlap(self, hp, hc):
        o = hp.inner(hc)
        return o * 4.0 * self.delta_f

def wf_wrapper(p):
    index = p['index']
    try:
        hp = gen.generate(**p)
        return index, hp
    except Exception as e:
        return index, None

def match_wrapper(p):
    '''A wrapper function to compute match
    '''
    h1 =pycbc.types.FrequencySeries(initial_array=p['h1_data'], delta_f=p['h1_delta_f'],epoch=p['h1_epoch'])
    h2 =pycbc.types.FrequencySeries(initial_array=p['h2_data'], delta_f=p['h2_delta_f'],epoch=p['h2_epoch'])
    return p['bank_index'], gen.match(h1, h2)

def gen_injections():
    mass_lim = (5, 100)
    spin_lim = (-0.5, 0.5)
    ecc_lim = (0, 0.3)
    ano_lim = (0, 2*np.pi)

    uniform_prior = pycbc.distributions.Uniform(
                            mass1=mass_lim,
                            mass2=mass_lim,
                            spin1z=spin_lim,
                            spin2z=spin_lim,
                            eccentricity=ecc_lim,
                            rel_anomaly=ano_lim)

    def _q_lt_8(params):
        return pycbc.conversions.q_from_mass1_mass2(params["mass1"],params["mass2"]) < 8

    return pycbc.distributions.JointDistribution(["mass1",
                                "mass2",
                                "spin1z",
                                "spin2z",
                                "eccentricity",
                                "rel_anomaly"],
                                uniform_prior,
                                constraints=[_q_lt_8])

In [2]:
gen = GenWaveform(buffer_length = 32, sample_rate = 2048, f_lower = 20)

In [3]:
with h5py.File('/work/yifanwang/ecc/bank/rollback/rerun-32buffer/buffer32eccbank.hdf') as f:
        df_bank = pd.DataFrame(
           {'mass1': f['mass1'][:],
            'mass2': f['mass2'][:],
            'tau0': pycbc.conversions.tau0_from_mass1_mass2(f['mass1'][:],f['mass2'][:],15),
            'eccentricity': f['eccentricity'][:],
            'rel_anomaly': f['rel_anomaly'][:],
            'spin1z': f['spin1z'][:],
            'spin2z': f['spin2z'][:],
            'approximant': f['approximant'][:].astype('str'),
            'f_lower': f['f_lower'][:]}
        )
df_bank['index'] = df_bank.index

In [4]:
inj = gen_injections()
df_ff = pd.DataFrame(inj.rvs(100))
df_ff['tau0'] = pycbc.conversions.tau0_from_mass1_mass2(df_ff['mass1'],df_ff['mass2'],15)
df_ff['index'] = df_ff.index
df_ff['approximant'] = df_bank['approximant'][0]
df_ff['f_lower'] = df_bank['f_lower'][0]

inj_cache = {}
parlist = ['index', 'approximant', 'f_lower', 'mass1', 'mass2', 'spin1z', 'spin2z', 'eccentricity', 'rel_anomaly']

with multiprocessing.Pool(128) as pool:
    for return_i, return_hp in pool.imap_unordered(
        wf_wrapper,
        ({k: df_ff.loc[idx,k] for k in parlist} for idx in tqdm(df_ff.index))
    ):
        inj_cache[return_i] = return_hp

100%|██████████| 100/100 [00:00<00:00, 1719.75it/s]
ERROR:pyseobnr.eob.dynamics.initial_conditions_aligned_ecc_opt:Internal function call failed: Input domain error. The predicted post-Newtonian initial separation (r0 = 6.840060289097139 M) is smaller than the minimum separation allowed (r_min = 7.0 M). Aborting the waveform generation since this region is outside the validity regime of the model. Please, review the physical sense of the input parameters.
ERROR:pyseobnr.eob.dynamics.initial_conditions_aligned_ecc_opt:Internal function call failed: Input domain error. The predicted post-Newtonian initial separation (r0 = 6.546287771146727 M) is smaller than the minimum separation allowed (r_min = 7.0 M). Aborting the waveform generation since this region is outside the validity regime of the model. Please, review the physical sense of the input parameters.
ERROR:pyseobnr.models.SEOBNRv5EHM:Waveform generation failed for q = 1.1428820147363625, chi_1 = 0.2739371643284165, chi_2 = 0.44207

In [10]:
df_ff['index']

0      0
1      1
2      2
3      3
4      4
      ..
95    95
96    96
97    97
98    98
99    99
Name: index, Length: 100, dtype: int64

In [12]:
all_fitting_factors = []
for ii in [0]:
    print(ii)
    hpinj = inj_cache[ii]
    if hpinj == None:
        print("Failed waveform generation in injections for #%i", ii)
        continue
    
    neighbor = df_bank[abs(df_bank['tau0']- df_ff.loc[ii,'tau0']) < 0.1].index
    print("Number of FF jobs = ", len(neighbor)) 

    wf_cache = {}    
    parlist = ['index', 'approximant', 'f_lower', 'mass1', 'mass2', 'spin1z', 'spin2z', 'eccentricity', 'rel_anomaly']
    with multiprocessing.Pool(128) as pool:
        for return_i, return_hp in pool.imap_unordered(
            wf_wrapper,
            ({k: df_bank.loc[idx,k] for k in parlist} for idx in tqdm(neighbor))
        ):
            wf_cache[return_i] = return_hp
    
    print("Template waveform generation done.") 
    calls = [
            {'bank_index': jj,
            'h1_data': hpinj.data,
            'h1_delta_f': hpinj.delta_f,
            'h1_epoch': hpinj.epoch,
            'h2_data': wf_cache[jj].data,
            'h2_delta_f': wf_cache[jj].delta_f,
            'h2_epoch': wf_cache[jj].epoch} for jj in neighbor
        ]
    
    maxmatch = 0
    maxindex = None
    print("Do some fitting factor calculations.") 
    with multiprocessing.Pool(128) as pool:
        for return_jj, return_match in pool.imap_unordered(
                match_wrapper,
                tqdm(calls)
                ):
            if return_match > maxmatch:
                maxmatch = return_match
                maxindex = return_jj

    dict_current = {'row': ii, 'fittingfactor': maxmatch}
    for cname in ['eccentricity', 'mass1', 'mass2', 'rel_anomaly', 'spin1z', 'spin2z', 'tau0']:
        dict_current['b'+cname] = df_bank.loc[maxindex, cname]
        
    all_fitting_factors += [dict_current]

0
Number of FF jobs =  3121


100%|██████████| 3121/3121 [00:28<00:00, 110.85it/s]


Template waveform generation done.
Do some fitting factor calculations.


100%|██████████| 3121/3121 [00:53<00:00, 57.83it/s] 


In [13]:
all_fitting_factors

[{'row': 0,
  'fittingfactor': 0.9959420561790466,
  'beccentricity': 0.2763540881310093,
  'bmass1': 31.18638626920827,
  'bmass2': 34.99931717983494,
  'brel_anomaly': 1.6599260646977783,
  'bspin1z': -0.1680280937071918,
  'bspin2z': -0.05469897999791412,
  'btau0': 1.752136419084542}]

In [14]:
all_fitting_factors = []
for ii in [0]:
    print(ii)
    hpinj = inj_cache[ii]
    if hpinj == None:
        print("Failed waveform generation in injections for #%i", ii)
        continue
    
    neighbor = df_bank[abs(df_bank['tau0']- df_ff.loc[ii,'tau0']) < 0.1].index
    print("Number of FF jobs = ", len(neighbor)) 

    wf_cache = {}    
    parlist = ['index', 'approximant', 'f_lower', 'mass1', 'mass2', 'spin1z', 'spin2z', 'eccentricity', 'rel_anomaly']
    with multiprocessing.Pool(128) as pool:
        for return_i, return_hp in pool.imap_unordered(
            wf_wrapper,
            ({k: df_bank.loc[idx,k] for k in parlist} for idx in tqdm(neighbor))
        ):
            wf_cache[return_i] = return_hp
    
    print("Template waveform generation done.") 
    calls = [
            {'bank_index': jj,
            'h1_data': hpinj.data,
            'h1_delta_f': hpinj.delta_f,
            'h1_epoch': hpinj.epoch,
            'h2_data': wf_cache[jj].data,
            'h2_delta_f': wf_cache[jj].delta_f,
            'h2_epoch': wf_cache[jj].epoch} for jj in neighbor
        ]
    
    maxmatch = 0
    maxindex = None
    print("Do some fitting factor calculations.") 
    for call in tqdm(calls):
        return_jj, return_match = match_wrapper(call)
        if return_match > maxmatch:
            maxmatch = return_match
            maxindex = return_jj

    dict_current = {'row': ii, 'fittingfactor': maxmatch}
    for cname in ['eccentricity', 'mass1', 'mass2', 'rel_anomaly', 'spin1z', 'spin2z', 'tau0']:
        dict_current['b'+cname] = df_bank.loc[maxindex, cname]
        
    all_fitting_factors += [dict_current]

0
Number of FF jobs =  3121


100%|██████████| 3121/3121 [00:28<00:00, 110.62it/s]


Template waveform generation done.
Do some fitting factor calculations.


100%|██████████| 3121/3121 [00:10<00:00, 304.36it/s]


In [15]:
all_fitting_factors

[{'row': 0,
  'fittingfactor': 0.9959420561790466,
  'beccentricity': 0.2763540881310093,
  'bmass1': 31.18638626920827,
  'bmass2': 34.99931717983494,
  'brel_anomaly': 1.6599260646977783,
  'bspin1z': -0.1680280937071918,
  'bspin2z': -0.05469897999791412,
  'btau0': 1.752136419084542}]